In [1]:
# Importações básicas
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Adiciona src ao path
sys.path.append('../src')

# Utilitários de dados
from data_utils import (
    load_raw_data,
    save_processed_data,
    remove_duplicates,
    handle_missing_values,
    detect_outliers,
    normalize_column,
    process_phone_string,
    process_phone_number,
    clean_and_lower_column,
    flatten_list_to_df,
    parse_utm_string,
    timestamp_ms_to_datetime
)

# Utilitários SQL
from sql_utils import DatabaseConnection as Dbc, load_query_from_file

# Utilitários de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitários de API
from api_utils import (
    make_request,
    get_json,
    post_json,
    paginated_request,
    response_to_dataframe
)

# utilitários hotmart
from hotmart_utils import Hotmart

# utilitários tmb
from tmb_utils import TMB  

# Utiliários Tally Forms
from tally_utils import Tally

# Configurações pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load Database Driver
db = Dbc()

# Inicializar API Hotmart
hotmart = Hotmart()

# Inicializar API TMB
tmb = TMB()

# Inicializar API Tally
tally = Tally()

print('✓ Importações concluídas com sucesso!')

✓ Importações concluídas com sucesso!


# Load Dataframes

In [2]:
df = db.execute_query_from_file('select_complete_lead_data_from_conversion')
df['phone'] = process_phone_string(df, 'phone')

df_buyers = db.execute_query_from_file('select_generic_sales', params={'product_name': 'Cronograma dos Fluentes'})

df_buyers['phone'] = process_phone_string(df_buyers, 'phone')

print('Df Shape buyers:', df_buyers.shape)

Df Shape buyers: (38988, 9)


# Second Buyers Verification

In [3]:
df_hotmart_pt_1 =  load_raw_data('hotmart_pt_1.csv', data_dir='../data/interim')

df_hotmart_pt_2 = load_raw_data('hotmart_pt_2.csv', data_dir='../data/interim')

df_hotmart_pt_3 = flatten_list_to_df(hotmart.get_sales_history(
    start_date='2025-12-01',
    end_date='2025-12-31',
))

df_hotmart_2 = pd.concat([df_hotmart_pt_1, df_hotmart_pt_2, df_hotmart_pt_3])


df_hotmart = load_raw_data('hotmart.csv', sep=';')

def clean_data(hotmart_transactions):
    # Drop columns: 'Nome do Produtor', 'Documento do Produtor' and 49 other columns
    hotmart_transactions = hotmart_transactions.drop(columns=['Nome do Produtor', 'Documento do Produtor', 'Nome do Afiliado', 'Transação', 'Meio de Pagamento', 'Origem', 'Moeda', 'Preço do Produto', 'Moeda.1', 'Preço da Oferta', 'Taxa de Câmbio', 'Moeda.2', 'Preço Original', 'Número da Parcela', 'Data de Venda', 'Data de Confirmação', 'Status', 'Documento', 'CEP', 'Cidade', 'Estado', 'Bairro', 'País', 'Endereço', 'Número', 'Complemento', 'chave', 'Código do Produto', 'Código da Afiliação', 'Tipo de Pagamento', 'Período Grátis', 'Tem co-produção', 'Venda feita como', 'Preço Total', 'Tipo pagamento oferta', 'Taxa de Câmbio Real', 'Preço Total Convertido', 'Quantidade de itens', 'Oferta de Upgrade', 'Cupom', 'Moeda.3', 'Valor que você recebeu convertido', 'Data Vencimento', 'Taxa de Câmbio do valor recebido', 'Instagram', 'Origem da venda', 'Moeda de recebimento', 'Faturamento líquido', 'Código do assinante', 'Nota Fiscal', 'Valor do frete bruto'])
    # Filter rows based on column: 'Recorrência'
    hotmart_transactions = hotmart_transactions[(hotmart_transactions['Recorrência'] == 1) | (hotmart_transactions['Recorrência'].isna())]
    hotmart_transactions['DDD'] = process_phone_string(hotmart_transactions, 'DDD')
    hotmart_transactions['Telefone'] = process_phone_string(hotmart_transactions, 'Telefone')
    hotmart_transactions['phone'] = hotmart_transactions.apply(lambda x: x['Telefone'] if pd.isna(x['DDD']) else f'55{x["DDD"]}{x["Telefone"]}', axis=1)
    # Drop column: 'DDD'
    hotmart_transactions = hotmart_transactions.drop(columns=['DDD', 'Telefone'])

    # Drop column: 'Recorrência'
    hotmart_transactions = hotmart_transactions.drop(columns=['Recorrência'])
    # Rename column 'Nome do Produto' to 'product_name'
    hotmart_transactions = hotmart_transactions.rename(columns={'Nome do Produto': 'product_name', "Nome": "name", "Email": "email","Código de Oferta": "offer_id" })
    # Drop column: 'Origem de Checkout'
    hotmart_transactions = hotmart_transactions.drop(columns=['Origem de Checkout'])
    hotmart_transactions = hotmart_transactions[hotmart_transactions['product_name'] == 'Cronograma dos Fluentes ']
    return hotmart_transactions

df_hotmart = clean_data(df_hotmart.copy())

df_tmb = load_raw_data('tmb.csv', sep=';')

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_tmb):
    # Drop columns: 'Produtor', 'Pedido' and 29 other columns
    df_tmb = df_tmb.drop(columns=['Produtor', 'Pedido', 'Endereço completo', 'Logradouro', 'Número', 'Bairro', 'Complemento', 'CEP', 'Estado', 'Cidade', 'País', 'Status', 'Status Financeiro', 'Status Cancelamento', 'Ticket (R$)', 'Modalidade de Contrato', 'Criado Em', 'Data Efetivado', 'Data Cancelado', 'utm_source', 'utm_medium', 'utm_campaign', 'utm_content', 'utm_last_source', 'utm_last_medium', 'utm_last_campaign', 'utm_last_content', 'Renovação automatica', 'Data de Renovação Prevista', 'Renovação enviada', 'Oferta'])
    # Drop column: 'Cliente CPF'
    df_tmb = df_tmb.drop(columns=['Cliente CPF'])
    # Rename column 'Produto' to 'offer_code'
    df_tmb = df_tmb.rename(columns={'Produto': 'offer_id', "Cliente Nome": "name", "Cliente Email": "email", "Telefone": "phone"})

    df_tmb['phone'] = process_phone_string(df_tmb, 'phone')
    
    return df_tmb

df_tmb = clean_data(df_tmb.copy())


"""
Cell generated by Data Wrangler.
"""
def clean_data(df_hotmart):
    # Filter rows based on column: 'purchase_recurrency_number'
    df_hotmart = df_hotmart[(df_hotmart['purchase_recurrency_number'] == 1) | (df_hotmart['purchase_recurrency_number'].isna())]
    # Drop columns: 'purchase_hotmart_fee_currency_code', 'purchase_hotmart_fee_total' and 22 other columns
    df_hotmart = df_hotmart.drop(columns=['purchase_hotmart_fee_currency_code', 'purchase_hotmart_fee_total', 'purchase_hotmart_fee_fixed', 'purchase_hotmart_fee_percentage', 'purchase_hotmart_fee_base', 'purchase_is_subscription', 'purchase_offer_payment_mode', 'purchase_tracking_source', 'purchase_payment_type', 'purchase_payment_installments_number', 'purchase_commission_as', 'purchase_warranty_expire_date', 'purchase_order_date', 'purchase_price_value', 'purchase_price_currency_code', 'producer_ucode', 'product_id', 'producer_name', 'buyer_ucode', 'purchase_tracking_source_sck', 'purchase_recurrency_number', 'purchase_tracking_external_code', 'purchase_payment_method', 'purchase_status'])
    # Rename column 'purchase_transaction' to 'transaction'
    df_hotmart = df_hotmart.rename(columns={'purchase_transaction': 'transaction'})
    # Rename column 'purchase_offer_code' to 'offer_code'
    df_hotmart = df_hotmart.rename(columns={'purchase_offer_code': 'offer_code'})
    # Rename column 'purchase_approved_date' to 'approved_at'
    df_hotmart = df_hotmart.rename(columns={'purchase_approved_date': 'approved_at'})
    # Rename column 'buyer_name' to 'name'
    df_hotmart = df_hotmart.rename(columns={'buyer_name': 'name'})
    # Rename column 'buyer_email' to 'email'
    df_hotmart = df_hotmart.rename(columns={'buyer_email': 'email'})
    # Rename column 'approved_at' to 'approval_date'
    df_hotmart = df_hotmart.rename(columns={'approved_at': 'approval_date'})
    df_hotmart['approval_date'] = timestamp_ms_to_datetime(df_hotmart, 'approval_date')
    return df_hotmart

df_hotmart_2 = clean_data(df_hotmart_2.copy())
df_hotmart_2.head()


df_tmb_2 = flatten_list_to_df(tmb.get_pedidos(page_size=1000))

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_tmb):
    # Filter rows based on column: 'status_pedido'
    df_tmb = df_tmb[df_tmb['status_pedido'] == "Efetivado"]
    # Drop columns: 'produtor', 'status_pedido' and 29 other columns
    df_tmb = df_tmb.drop(columns=['produtor', 'status_pedido', 'documento', 'valor_total', 'parcelas', 'valor_entrada', 'valor_parcela', 'taxa_administracao', 'endereco_completo', 'criado_em', 'telefone', 'produto_id', 'nascimento', 'produtor_id', 'utm_source', 'utm_medium', 'utm_content', 'utm_campaign', 'utm_last_source', 'melhor_dia_pagamento', 'utm_last_medium', 'utm_last_content', 'utm_last_campaign', 'pais', 'cep', 'endereco_estado', 'endereco_cidade', 'endereco_bairro', 'endereco_logradouro', 'endereco_numero', 'endereco_complemento'])
    # Rename column 'lancamento' to 'offer_code'
    df_tmb = df_tmb.rename(columns={'lancamento': 'offer_code'})
    # Rename column 'pedido_id' to 'transaction'
    df_tmb = df_tmb.rename(columns={'pedido_id': 'transaction'})
    # Rename column 'cliente' to 'name'
    df_tmb = df_tmb.rename(columns={'cliente': 'name'})
    # Rename column 'data_efetivado' to 'approval_date'
    df_tmb = df_tmb.rename(columns={'data_efetivado': 'approval_date'})
    # Change column type to datetime64[ns] for column: 'approval_date'
    df_tmb = df_tmb.astype({'approval_date': 'datetime64[ns]'})
    return df_tmb

df_tmb_2 = clean_data(df_tmb_2.copy())
df_tmb_2.head()

✓ Concluído: 21 página(s) processada(s). Total de 1018 itens obtidos.)
✓ Concluído: 1 página(s) processada(s). Total de 100 pedidos obtidos.


,offer_code,transaction,name,email,approval_date,status_financeiro
0,COM-BF25-alunos-Oferta Negociação,1918767,Matheus Ruas,matheusruas@hotmail.com,2025-12-02 18:53:01.355239,Adimplente
1,COM-BF25-alunos-Oferta Negociação,1917023,KLEBER FERREIRA DO VALE,klebervaleadvogado@outlook.com,2025-12-02 19:00:26.471830,Adimplente
2,COM-BF25-alunos-Oferta Negociação,1915079,Kyara Costa de Azevedo,kyky338.157@gmail.com,2025-12-01 10:55:35.356717,Adimplente
3,COM-BF25-Oferta Downsell-3,1914968,Bruno Luis Reis Intaschi,brunointaschi@hotmail.com,2025-12-01 11:21:29.853887,Adimplente
4,COM-BF25-Oferta Downsell-3,1914804,Joana Darc Santos Almeida Oliveira,oliveiragiseli071@gmail.com,2025-12-01 09:30:20.814242,Adimplente


## Concat keys to Exclude

In [4]:
# Lead Ids Set
lead_ids_to_exclude = set()
lead_ids_to_exclude = lead_ids_to_exclude.union(set(df_buyers['lead_id'].unique()))

# Emails Set
emails_to_exclude = set()
emails_to_exclude = emails_to_exclude.union(set(df_buyers['email'].dropna().unique()))
emails_to_exclude = emails_to_exclude.union(set(df_tmb['email'].dropna().unique()))
emails_to_exclude = emails_to_exclude.union(set(df_hotmart['email'].dropna().unique()))
emails_to_exclude = emails_to_exclude.union(set(df_hotmart_2['email'].dropna().unique()))
emails_to_exclude = emails_to_exclude.union(set(df_tmb_2['email'].dropna().unique()))

# Phone Set
phones_to_exclude = set()
phones_to_exclude = phones_to_exclude.union(set(df_buyers['phone'].unique()))
phones_to_exclude = phones_to_exclude.union(set(df_tmb['phone'].unique()))
phones_to_exclude = phones_to_exclude.union(set(df_hotmart['phone'].unique()))

## Exclude leads from main_df

In [5]:
df = df.drop_duplicates(subset=['lead_id'])
# Drop duplicate rows in column: 'email'
df = df.drop_duplicates(subset=['email'])
# Drop duplicate rows in column: 'phone'
df = df.drop_duplicates(subset=['phone'])
# Drop rows with missing phone
df = df.dropna(subset=['phone'])

In [6]:
df = df[~df['lead_id'].isin(lead_ids_to_exclude)]
df = df[~df['email'].isin(emails_to_exclude)]
df = df[~df['phone'].isin(phones_to_exclude)]

print('Df Shape after exclusion:', df.shape)

Df Shape after exclusion: (227567, 13)


In [7]:
# Qualificados gerais
df_multifatorial = df[df['qualificado_geral'] == True]

# Renda qualificada: não qualificado_geral, mas renda_qualificada = True, removendo os já qualificados
df_renda_qualificada = df[
    (df['qualificado_geral'] == False) & 
    (df['renda_qualificada'] == True) & 
    (~df['lead_id'].isin(df_multifatorial['lead_id']))
]

# Desqualificados: não qualificado_geral nem renda_qualificada, removendo já selecionados acima
df_renda_desqualificada = df[
    (df['qualificado_geral'] == False) & 
    (df['renda_qualificada'] == False) & 
    (~df['lead_id'].isin(df_multifatorial['lead_id'])) & 
    (~df['lead_id'].isin(df_renda_qualificada['lead_id']))
]

save_processed_data(df_multifatorial, 'prospect_bf25fo-leads_multifatorial.csv')
save_processed_data(df_renda_qualificada, 'prospect_bf25fo-leads_renda_qualificada.csv')
save_processed_data(df_renda_desqualificada, 'prospect_bf25fo-leads_renda_desqualificada.csv')

print("Multifatorial:", df_multifatorial.shape)
print("Renda Qualificada:", df_renda_qualificada.shape)
print("Renda Desqualificada:", df_renda_desqualificada.shape)

Multifatorial: (28510, 13)
Renda Qualificada: (20472, 13)
Renda Desqualificada: (178585, 13)
